Explore Spotify Releases.

- Does every entry have a title, artist, release date and ISRC?
- What are the earliest and latest release dates?
- How many fall within the study window?
- Are any recordings listed more than once?
- Do a few artists account for many of the results?

## DataFrame reference

Reference for the current dataset after running the notebook from top to bottom. Update this table when the data or preparation steps change.

| DataFrame | What one row represents | Contents / purpose | Created from |
|---|---|---|---|
| `tracks_df` | One Spotify track entry | All 100 entries, original nested metadata, and extracted ISRC | JSON `data["tracks"]` |
| `album_df` | Album information for one track entry | Expanded album fields and converted release dates; 100 rows, not necessarily unique albums | `tracks_df["album"]` |
| `filtered_album_df` | Album information for one eligible track | 68 entries within January 1–September 1, 2026, inclusive; original indexes retained | Date-filtered `album_df` |
| `filtered_tracks_df` | One eligible track entry | The matching 68 tracks with nested metadata and ISRCs; converted dates remain in the album table | `tracks_df`, selected using `filtered_album_df.index` |
| `track_artist_df` | One track–artist credit | 143 rows with `track_id`, `artist_id`, and `artist_name`; 133 unique artist IDs | Expanded artist lists from `filtered_tracks_df` |
| `stream_test_df` | One track selected for a stream-visibility check | First 5 eligible tracks (not random); `id`, `name`, `artists`, `external_urls`, and extracted `spotify_url` | First five rows of `filtered_tracks_df` |
| `stream_observations_df` | One track observation from the batch test | Current test: 3 tracks; `track_id`, `track_name`, `artist`, `search_genre` (source query), nullable integer `stream_count`, UTC `observation_datetime`, and retrieval `status` | `batch_retrieve_stream_counts(stream_test_df.head(3))` |

**Rebuilding tables:** After changing an earlier preparation step, rerun the downstream cells—or restart the kernel and Run All. Existing derived DataFrames do not update automatically.

**Index alignment:** The track and album tables share their original row indexes. Preserve those indexes until the matching filtered tracks have been selected. The artist-credit table uses a new index and retains `track_id` to link credits back to tracks.


## 1. Import libraries

In [2]:
import json
import pandas as pd

## 2. Load data and create a DataFrame

In [3]:
# import json file, using utf-8 for translating a file's
# bytes to characters, save as data
with open("data/100_edm_tracks.json", encoding="utf-8") as file: 
    data = json.load(file)

tracks_df = pd.DataFrame(data["tracks"])
tracks_df.head()


,album,artists,disc_number,duration_ms,explicit,external_ids,external_urls,href,id,is_local,is_playable,name,track_number,type,uri
0,"{'album_type': 'single', 'artists': [{'externa...",[{'external_urls': {'spotify': 'https://open.s...,1,127846,False,{'isrc': 'QT3F92642024'},{'spotify': 'https://open.spotify.com/track/2s...,https://api.spotify.com/v1/tracks/2sxYN584YAgn...,2sxYN584YAgnRwnmIQF0Ye,False,True,CRECIENDO DIARIAMENTE - Afro House,3,track,spotify:track:2sxYN584YAgnRwnmIQF0Ye
1,"{'album_type': 'single', 'artists': [{'externa...",[{'external_urls': {'spotify': 'https://open.s...,1,162804,False,{'isrc': 'DK4YA2600951'},{'spotify': 'https://open.spotify.com/track/0f...,https://api.spotify.com/v1/tracks/0fS2ZLRBEehM...,0fS2ZLRBEehMSTUXyy78mM,False,True,Losing My Religion,1,track,spotify:track:0fS2ZLRBEehMSTUXyy78mM
2,"{'album_type': 'single', 'artists': [{'externa...",[{'external_urls': {'spotify': 'https://open.s...,1,133250,False,{'isrc': 'QZQAY2668235'},{'spotify': 'https://open.spotify.com/track/2P...,https://api.spotify.com/v1/tracks/2PuZF8KqwkGN...,2PuZF8KqwkGN9eEatQ1xDs,False,True,Mi Chico - Josh Fawaz Remix,3,track,spotify:track:2PuZF8KqwkGN9eEatQ1xDs
3,"{'album_type': 'single', 'artists': [{'externa...",[{'external_urls': {'spotify': 'https://open.s...,1,183870,False,{'isrc': 'QZTPX2680490'},{'spotify': 'https://open.spotify.com/track/65...,https://api.spotify.com/v1/tracks/65cd1kUvEU9Q...,65cd1kUvEU9Q0PmYJRc6Uy,False,True,Carne y Tambor,1,track,spotify:track:65cd1kUvEU9Q0PmYJRc6Uy
4,"{'album_type': 'single', 'artists': [{'externa...",[{'external_urls': {'spotify': 'https://open.s...,1,196500,False,{'isrc': 'AUNV02600073'},{'spotify': 'https://open.spotify.com/track/7v...,https://api.spotify.com/v1/tracks/7vY4hbChNHJu...,7vY4hbChNHJudQQpPlnL5M,False,True,Deeper Love,1,track,spotify:track:7vY4hbChNHJudQQpPlnL5M


## 3. Initial data inspection

In [4]:
# rows, columns, datatypes
tracks_df.info()


<class 'pandas.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 15 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   album          100 non-null    object
 1   artists        100 non-null    object
 2   disc_number    100 non-null    int64 
 3   duration_ms    100 non-null    int64 
 4   explicit       100 non-null    bool  
 5   external_ids   100 non-null    object
 6   external_urls  100 non-null    object
 7   href           100 non-null    str   
 8   id             100 non-null    str   
 9   is_local       100 non-null    bool  
 10  is_playable    100 non-null    bool  
 11  name           100 non-null    str   
 12  track_number   100 non-null    int64 
 13  type           100 non-null    str   
 14  uri            100 non-null    str   
dtypes: bool(3), int64(3), object(4), str(5)
memory usage: 9.8+ KB


In [5]:
# count missing values in top-level columns
tracks_df.isna().sum()

album            0
artists          0
disc_number      0
duration_ms      0
explicit         0
external_ids     0
external_urls    0
href             0
id               0
is_local         0
is_playable      0
name             0
track_number     0
type             0
uri              0
dtype: int64

The dataset contains 100 entries and 15 columns. No top-level missing values were detected. Nested fields still need inspection.

## 4. Prepare fields for analysis

In [6]:
# create a new column with isrc code, using the external_ids dictionary
tracks_df["isrc"] = tracks_df["external_ids"].apply(
    lambda x: x.get("isrc") if isinstance(x, dict) else None
)

# check for null values in the new isrc column
tracks_df["isrc"].isna().sum()

np.int64(0)

In [7]:
# check for empty nested fields next
# Expand album dictionaries into a table
album_df = pd.json_normalize(tracks_df["album"])

# Count missing values in the resulting columns
album_df.isna().sum()

album_type                0
artists                   0
href                      0
id                        0
images                    0
is_playable               0
name                      0
release_date              0
release_date_precision    0
total_tracks              0
type                      0
uri                       0
external_urls.spotify     0
dtype: int64

In [8]:
# check whether release dates are usable for analysis
print(f"Release date value counts:\n", album_df[["release_date"]].value_counts(), sep='')

Release date value counts:
release_date
2026-03-06      5
2026-09-25      5
2026-05-29      5
2026-10-30      4
2026-06-08      3
2026-09-18      3
2026-08-21      3
2026-10-09      3
2026-05-08      3
2026-03-13      3
2026-08-28      3
2026-04-03      3
2026-10-16      3
2026-10-02      3
2026-09-04      2
2026-10-24      2
2026-02-06      2
2026-05-15      2
2026-01-23      2
2026-03-26      2
2026-03-27      2
2026-07-24      2
2026-06-26      2
2026-09-11      2
2026-04-17      2
2026-07-03      2
2026-05-22      2
2026-05-01      2
2026-01-15      1
2026-06-19      1
2026-10-25      1
2026-06-14      1
2026-02-27      1
2026-10-15      1
2026-04-24      1
2026-06-12      1
2026-08-07      1
2026-04-10      1
2026-06-05      1
2026-10-17      1
2026-08-26      1
2026-04-29      1
2026-07-18      1
2026-10-26      1
2026-01-29      1
2026-01-09      1
2026-07-14      1
2026-10-23      1
2026-03-05      1
2026-01-24      1
2026-04-28      1
Name: count, dtype: int64


In [9]:
# sanity check
album_df.head(5)

,album_type,artists,href,id,images,is_playable,name,release_date,release_date_precision,total_tracks,type,uri,external_urls.spotify
0,single,[{'external_urls': {'spotify': 'https://open.s...,https://api.spotify.com/v1/albums/5jKfzHgw2kUj...,5jKfzHgw2kUj2GgzZZyXDV,"[{'height': 640, 'width': 640, 'url': 'https:/...",True,POR MáS DíAS DE BIEN,2026-06-08,day,3,album,spotify:album:5jKfzHgw2kUj2GgzZZyXDV,https://open.spotify.com/album/5jKfzHgw2kUj2Gg...
1,single,[{'external_urls': {'spotify': 'https://open.s...,https://api.spotify.com/v1/albums/2tBXlHUu40fy...,2tBXlHUu40fyP9eKw13D1L,"[{'height': 640, 'width': 640, 'url': 'https:/...",True,Losing My Religion,2026-09-04,day,1,album,spotify:album:2tBXlHUu40fyP9eKw13D1L,https://open.spotify.com/album/2tBXlHUu40fyP9e...
2,single,[{'external_urls': {'spotify': 'https://open.s...,https://api.spotify.com/v1/albums/5BskxQl78jot...,5BskxQl78jotslv7DMPW4p,"[{'height': 640, 'width': 640, 'url': 'https:/...",True,Mi Chico (Josh Fawaz Remix),2026-10-24,day,3,album,spotify:album:5BskxQl78jotslv7DMPW4p,https://open.spotify.com/album/5BskxQl78jotslv...
3,single,[{'external_urls': {'spotify': 'https://open.s...,https://api.spotify.com/v1/albums/4UCEjU56lQmW...,4UCEjU56lQmWaAIc6bxxea,"[{'height': 640, 'width': 640, 'url': 'https:/...",True,Carne y Tambor,2026-02-06,day,1,album,spotify:album:4UCEjU56lQmWaAIc6bxxea,https://open.spotify.com/album/4UCEjU56lQmWaAI...
4,single,[{'external_urls': {'spotify': 'https://open.s...,https://api.spotify.com/v1/albums/2ZwcQ15i7nzK...,2ZwcQ15i7nzKnXh5zIkhhO,"[{'height': 640, 'width': 640, 'url': 'https:/...",True,Deeper Love,2026-05-15,day,1,album,spotify:album:2ZwcQ15i7nzKnXh5zIkhhO,https://open.spotify.com/album/2ZwcQ15i7nzKnXh...


In [10]:
# validate dates are in a usable format
album_df["release_date"] = pd.to_datetime(
    album_df["release_date"], 
    errors="coerce")

# check for missing values in the release_date column
print(f"Missing Values: ", album_df["release_date"].isna().sum())

#
print(f"Data type: ", album_df["release_date"].dtypes)


Missing Values:  0
Data type:  datetime64[us]


## 5. Select the study window

In [11]:
# filter dates to only inlude January 1, 2026 - September 1, 2026
# make working copy of the filtered dataframe to avoid SettingWithCopyWarning

filtered_album_df = album_df[(
    album_df["release_date"] >= "2026-01-01") & 
    (album_df["release_date"] <= "2026-09-01")].copy()

filtered_album_df.head(5)

,album_type,artists,href,id,images,is_playable,name,release_date,release_date_precision,total_tracks,type,uri,external_urls.spotify
0,single,[{'external_urls': {'spotify': 'https://open.s...,https://api.spotify.com/v1/albums/5jKfzHgw2kUj...,5jKfzHgw2kUj2GgzZZyXDV,"[{'height': 640, 'width': 640, 'url': 'https:/...",True,POR MáS DíAS DE BIEN,2026-06-08,day,3,album,spotify:album:5jKfzHgw2kUj2GgzZZyXDV,https://open.spotify.com/album/5jKfzHgw2kUj2Gg...
3,single,[{'external_urls': {'spotify': 'https://open.s...,https://api.spotify.com/v1/albums/4UCEjU56lQmW...,4UCEjU56lQmWaAIc6bxxea,"[{'height': 640, 'width': 640, 'url': 'https:/...",True,Carne y Tambor,2026-02-06,day,1,album,spotify:album:4UCEjU56lQmWaAIc6bxxea,https://open.spotify.com/album/4UCEjU56lQmWaAI...
4,single,[{'external_urls': {'spotify': 'https://open.s...,https://api.spotify.com/v1/albums/2ZwcQ15i7nzK...,2ZwcQ15i7nzKnXh5zIkhhO,"[{'height': 640, 'width': 640, 'url': 'https:/...",True,Deeper Love,2026-05-15,day,1,album,spotify:album:2ZwcQ15i7nzKnXh5zIkhhO,https://open.spotify.com/album/2ZwcQ15i7nzKnXh...
5,album,[{'external_urls': {'spotify': 'https://open.s...,https://api.spotify.com/v1/albums/1aHWSGAetL13...,1aHWSGAetL13Kkl2amjrnQ,"[{'height': 640, 'width': 640, 'url': 'https:/...",True,SOMA,2026-03-06,day,21,album,spotify:album:1aHWSGAetL13Kkl2amjrnQ,https://open.spotify.com/album/1aHWSGAetL13Kkl...
6,album,[{'external_urls': {'spotify': 'https://open.s...,https://api.spotify.com/v1/albums/1aHWSGAetL13...,1aHWSGAetL13Kkl2amjrnQ,"[{'height': 640, 'width': 640, 'url': 'https:/...",True,SOMA,2026-03-06,day,21,album,spotify:album:1aHWSGAetL13Kkl2amjrnQ,https://open.spotify.com/album/1aHWSGAetL13Kkl...


In [12]:
# confirm the count of rows in the filtered dataframe is 
# less than the original dataframe
print(f"Original DataFrame rows: {len(album_df)}")
print(f"Filtered DataFrame rows: {len(filtered_album_df)}")

Original DataFrame rows: 100
Filtered DataFrame rows: 68


In [13]:
# apply the same date selection to the original tracks_df dataframe, 
# using the index of the filtered_album_df
# make working copy to save

filtered_tracks_df = tracks_df.loc[filtered_album_df.index].copy()

print(f"Filtered Tracks DataFrame rows: {len(filtered_tracks_df)}")

# check row 7 of both dataframes to confirm they are the same
print(f"Filtered Album DataFrame row 7:\n", filtered_album_df["name"].iloc[7])
print(f"Filtered Tracks DataFrame row 7:\n", filtered_tracks_df["name"].iloc[7])

filtered_tracks_df.head(5)

Filtered Tracks DataFrame rows: 68
Filtered Album DataFrame row 7:
 Memeza, Pt. 1
Filtered Tracks DataFrame row 7:
 Memeza, Pt. 1


,album,artists,disc_number,duration_ms,explicit,external_ids,external_urls,href,id,is_local,is_playable,name,track_number,type,uri,isrc
0,"{'album_type': 'single', 'artists': [{'externa...",[{'external_urls': {'spotify': 'https://open.s...,1,127846,False,{'isrc': 'QT3F92642024'},{'spotify': 'https://open.spotify.com/track/2s...,https://api.spotify.com/v1/tracks/2sxYN584YAgn...,2sxYN584YAgnRwnmIQF0Ye,False,True,CRECIENDO DIARIAMENTE - Afro House,3,track,spotify:track:2sxYN584YAgnRwnmIQF0Ye,QT3F92642024
3,"{'album_type': 'single', 'artists': [{'externa...",[{'external_urls': {'spotify': 'https://open.s...,1,183870,False,{'isrc': 'QZTPX2680490'},{'spotify': 'https://open.spotify.com/track/65...,https://api.spotify.com/v1/tracks/65cd1kUvEU9Q...,65cd1kUvEU9Q0PmYJRc6Uy,False,True,Carne y Tambor,1,track,spotify:track:65cd1kUvEU9Q0PmYJRc6Uy,QZTPX2680490
4,"{'album_type': 'single', 'artists': [{'externa...",[{'external_urls': {'spotify': 'https://open.s...,1,196500,False,{'isrc': 'AUNV02600073'},{'spotify': 'https://open.spotify.com/track/7v...,https://api.spotify.com/v1/tracks/7vY4hbChNHJu...,7vY4hbChNHJudQQpPlnL5M,False,True,Deeper Love,1,track,spotify:track:7vY4hbChNHJudQQpPlnL5M,AUNV02600073
5,"{'album_type': 'album', 'artists': [{'external...",[{'external_urls': {'spotify': 'https://open.s...,1,206000,False,{'isrc': 'GXBAV2567659'},{'spotify': 'https://open.spotify.com/track/53...,https://api.spotify.com/v1/tracks/53IpJlqruSht...,53IpJlqruShtCrrs09bqH1,False,True,AMADO,10,track,spotify:track:53IpJlqruShtCrrs09bqH1,GXBAV2567659
6,"{'album_type': 'album', 'artists': [{'external...",[{'external_urls': {'spotify': 'https://open.s...,1,236000,False,{'isrc': 'QT5M22515698'},{'spotify': 'https://open.spotify.com/track/1y...,https://api.spotify.com/v1/tracks/1y2plKQTw6Py...,1y2plKQTw6PyGHxTmVJpuM,False,True,ALGORA [MAQUINA],17,track,spotify:track:1y2plKQTw6PyGHxTmVJpuM,QT5M22515698


## 6. Inspect duplicate records

In [14]:
# check for duplicates in the filtered tracks dataframe
print(f"Duplicate rows in filtered_tracks_df: ", filtered_tracks_df["isrc"].duplicated().sum())

Duplicate rows in filtered_tracks_df:  0


## 7. Examine artist credits

Expand artist credits and count unique artist IDs, including collaborators.

In [15]:
# inspect one artist name to see if it is a string or a list
print(f"Artist inspection: ", filtered_tracks_df["artists"].iloc[0])

Artist inspection:  [{'external_urls': {'spotify': 'https://open.spotify.com/artist/204dOGG0fctk51d79aqtJG'}, 'href': 'https://api.spotify.com/v1/artists/204dOGG0fctk51d79aqtJG', 'id': '204dOGG0fctk51d79aqtJG', 'name': 'Cokito Deep', 'type': 'artist', 'uri': 'spotify:artist:204dOGG0fctk51d79aqtJG'}]


In [16]:
# create a new DataFrame with one row per track-artist credit
track_artist_df = (
    filtered_tracks_df[["id", "artists"]]
    .explode("artists") # turns each item in a list to a separate row
    .reset_index(drop=True)
)
track_artist_df.head(5)
track_artist_df.loc[0, "artists"]

{'external_urls': {'spotify': 'https://open.spotify.com/artist/204dOGG0fctk51d79aqtJG'},
 'href': 'https://api.spotify.com/v1/artists/204dOGG0fctk51d79aqtJG',
 'id': '204dOGG0fctk51d79aqtJG',
 'name': 'Cokito Deep',
 'type': 'artist',
 'uri': 'spotify:artist:204dOGG0fctk51d79aqtJG'}

In [17]:
# extract artist ID and names into their own columns
track_artist_df["artist_id"] = track_artist_df["artists"].apply(
    lambda artist: artist.get("id")
)

track_artist_df["artist_name"] = track_artist_df["artists"].apply(
    lambda artist: artist.get("name")
)

track_artist_df.head(5)

,id,artists,artist_id,artist_name
0,2sxYN584YAgnRwnmIQF0Ye,{'external_urls': {'spotify': 'https://open.sp...,204dOGG0fctk51d79aqtJG,Cokito Deep
1,65cd1kUvEU9Q0PmYJRc6Uy,{'external_urls': {'spotify': 'https://open.sp...,00CMSJdbf36zOzKB3z8JrR,Victor Cardenas
2,65cd1kUvEU9Q0PmYJRc6Uy,{'external_urls': {'spotify': 'https://open.sp...,14ASSasIb42tG16ym4kVfh,Exotic
3,65cd1kUvEU9Q0PmYJRc6Uy,{'external_urls': {'spotify': 'https://open.sp...,62mc7565AIVwculibZaNOZ,Konig Pry
4,65cd1kUvEU9Q0PmYJRc6Uy,{'external_urls': {'spotify': 'https://open.sp...,4IrirmbSt5yUzaeQ0wvO0F,Ricardo RV


In [18]:
# rename track ID column and keep the three fields
track_artist_df = track_artist_df.rename(columns={"id": "track_id"})[["track_id", "artist_id", "artist_name"]]
track_artist_df[["track_id", "artist_id", "artist_name"]]
track_artist_df.head(5)

,track_id,artist_id,artist_name
0,2sxYN584YAgnRwnmIQF0Ye,204dOGG0fctk51d79aqtJG,Cokito Deep
1,65cd1kUvEU9Q0PmYJRc6Uy,00CMSJdbf36zOzKB3z8JrR,Victor Cardenas
2,65cd1kUvEU9Q0PmYJRc6Uy,14ASSasIb42tG16ym4kVfh,Exotic
3,65cd1kUvEU9Q0PmYJRc6Uy,62mc7565AIVwculibZaNOZ,Konig Pry
4,65cd1kUvEU9Q0PmYJRc6Uy,4IrirmbSt5yUzaeQ0wvO0F,Ricardo RV


In [19]:
# count unique artists in the filtered tracks dataframe
# nuunique() returns the number of unique values in a Series
unique_artists_count = track_artist_df["artist_id"].nunique()
unique_artists_count

133

## 8. Test public stream-count visibility

Prepare links for the first five eligible tracks. This is a visibility test, not a random sample or a stream-count collection result. Record the exact display and observation date when inspecting Spotify; a blank remains “not displayed,” not confirmed under 1,000.

In [49]:
# research streaming capabilities
# show five tracks and their links in the notebook

# select the first five rows of the filtered_tracks_df dataframe and keep only the id, name, and external_urls columns
stream_test_df = filtered_tracks_df[["id", "name", "artists", "external_urls"]].head(5).copy()

# create a new column with the Spotify URL, using the external_urls dictionary
stream_test_df["spotify_url"] = stream_test_df["external_urls"].apply(
    lambda urls: urls.get("spotify")
)

# display the id, name, and spotify_url columns of the stream_test_df dataframe
stream_test_df[["id", "name", "spotify_url"]]

,id,name,spotify_url
0,2sxYN584YAgnRwnmIQF0Ye,CRECIENDO DIARIAMENTE - Afro House,https://open.spotify.com/track/2sxYN584YAgnRwn...
3,65cd1kUvEU9Q0PmYJRc6Uy,Carne y Tambor,https://open.spotify.com/track/65cd1kUvEU9Q0Pm...
4,7vY4hbChNHJudQQpPlnL5M,Deeper Love,https://open.spotify.com/track/7vY4hbChNHJudQQ...
5,53IpJlqruShtCrrs09bqH1,AMADO,https://open.spotify.com/track/53IpJlqruShtCrr...
6,1y2plKQTw6PyGHxTmVJpuM,ALGORA [MAQUINA],https://open.spotify.com/track/1y2plKQTw6PyGHx...


In [21]:
# display one complete URL for a track to confirm it works
stream_test_df["spotify_url"].iloc[0]

'https://open.spotify.com/track/2sxYN584YAgnRwnmIQF0Ye'

### Observation 1 — November 23, 2026

The first test link opened successfully. The displayed play count for **CRECIENDO DIARIAMENTE - Afro House** by **Cokito Deep** was **4,592,700**, placing this observation above 1,000 plays.

This establishes that a numerical count was visible for this track. It does not yet establish automated collection feasibility or how to interpret blank displays. The count was observed in the Spotify web player.

The running record of individual observations is maintained in [observations.md](observations.md). Use that table as the observation log; the notebook provides context and interpretation.


In [22]:
# check 2nd track url to confirm it works
stream_test_df["spotify_url"].iloc[1]

'https://open.spotify.com/track/65cd1kUvEU9Q0PmYJRc6Uy'

### Observation 2 — November 23, 2026

**Carne y Tambor:** 2,418,105 displayed plays in the web player. Link opened successfully.


In [23]:
# check 3rd track url to confirm it works
stream_test_df["spotify_url"].iloc[2]

'https://open.spotify.com/track/7vY4hbChNHJudQQpPlnL5M'

### Observation 3 — November 23, 2026

**Deeper Love:** 1,704,503 displayed plays in the web player.


In [24]:
# check 4th track url to confirm it works
stream_test_df["spotify_url"].iloc[3]

'https://open.spotify.com/track/53IpJlqruShtCrrs09bqH1'

### Observation 4 — November 23, 2026

**AMADO:** 7,183,220 displayed plays in the Spotify web player.


In [25]:
# check 5th track url to confirm it works
stream_test_df["spotify_url"].iloc[4]

'https://open.spotify.com/track/1y2plKQTw6PyGHxTmVJpuM'

### Observation 5 — November 23, 2026

**ALGORA [MAQUINA]:** 4,355,925 displayed plays in the Spotify web player.

All five inspected tracks exceed one million displayed plays. These were the first five eligible search results, not a random sample. This confirms visible counts for these tracks but does not establish access to low-stream tracks or a representative sample.


## 9. Create a function to open a track link and return its stream count

In [28]:
pip install bs4

  Using cached bs4-0.0.2-py2.py3-none-any.whl.metadata (411 bytes)
Using cached bs4-0.0.2-py2.py3-none-any.whl (1.2 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [bs4]

[notice] A new release of pip is available: 25.3 -> 26.3
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [31]:
# Create a function that accepts one Spotify track URL from stream_test_df.
# Try to retrieve the displayed stream count.
# Return the count if available, or a status explaining why it wasn't retrieved.
# Do not treat a blank or missing count as zero.

from urllib import response


def get_stream_count(spotify_url):
    # import requests and BeautifulSoup for web scraping
    import requests
    from bs4 import BeautifulSoup

    # send a GET request to the Spotify track URL
    response = requests.get(spotify_url)

    print("Returned URL:", response.url)
    print("Page preview:", response.text[:500])
    print("Contains playcount:", "playcount" in response.text)

    # check if the request was successful (status code 200)
    if response.status_code == 200:
        # parse the HTML content of the page
        soup = BeautifulSoup(response.content, 'html.parser')

        # find the element that contains the stream count
        stream_count_element = soup.find("span", attrs={"data-testid": "playcount"})  # Replace with actual class name

        # check if the stream count element was found
        if stream_count_element:
            return stream_count_element.text.strip()  # Return the text of the stream count d
        else:
            return "Stream count not found on page."
    else:
        return f"Failed to retrieve page. Status code: {response.status_code}"

    
get_stream_count(stream_test_df.iloc[1]["spotify_url"])




Returned URL: https://open.spotify.com/track/65cd1kUvEU9Q0PmYJRc6Uy
Page preview: <!DOCTYPE html><html class="mobile-web-player" lang="en" dir="ltr"><head><meta charSet="utf-8"/><title>Carne y Tambor - song and lyrics by Victor Cardenas, Exotic, Konig Pry, Ricardo RV | Spotify</title><link rel="icon" sizes="32x32" type="image/png" href="https://open.spotifycdn.com/cdn/images/favicon32.b64ecc03.png"/><link rel="icon" sizes="16x16" type="image/png" href="https://open.spotifycdn.com/cdn/images/favicon16.1c487bff.png"/><link rel="icon" href="https://open.spotifycdn.com/cdn/images
Contains playcount: False


'Stream count not found on page.'

In [33]:
# playwright is a tool that lets Python control a real web browser
pip install playwright

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.8/48.8 MB 65.8 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [playwright]2 [playwright]

[notice] A new release of pip is available: 25.3 -> 26.3
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [40]:
import sys
import subprocess

subprocess.run([sys.executable, "-m", "pip", "install", "playwright"],
               check=True
               )

subprocess.run([sys.executable, "-m", "playwright", "install", "chromium"])



[notice] A new release of pip is available: 25.3 -> 26.3
[notice] To update, run: pip3 install --upgrade pip


|                                                                                |   0% of 191.9 MiB
|■■■■■■■■                                                                        |  10% of 191.9 MiB
|■■■■■■■■■■■■■■■■                                                                |  20% of 191.9 MiB
|■■■■■■■■■■■■■■■■■■■■■■■■                                                        |  30% of 191.9 MiB
|■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■                                                |  40% of 191.9 MiB
|■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■                                        |  50% of 191.9 MiB
|■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■                                |  60% of 191.9 MiB
|■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■                        |  70% of 191.9 MiB
|■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■                |  80% of 191.9 MiB
|■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■        |  90% of 

CompletedProcess(args=['/usr/local/bin/python3', '-m', 'playwright', 'install', 'chromium'], returncode=0)

In [41]:
# After requests did not retrieve the stream count, test browser automation with Playwright.
from playwright.async_api import async_playwright, TimeoutError as PlaywrightTimeoutError

async def get_stream_count_browser(spotify_url):
    async with async_playwright() as p:
        # Open Chromium so we can watch the test
        browser = await p.chromium.launch(headless=False)

        try:
            page = await browser.new_page()
            await page.goto(spotify_url, wait_until="domcontentloaded")

            # Wait up to 15 seconds for the play-count element
            playcount = page.locator('[data-testid="playcount"]').first
            await playcount.wait_for(state="visible", timeout=15000)

            return (await playcount.inner_text()).strip()

        except PlaywrightTimeoutError:
            return "Page or stream count did not load within the time limit."

        finally:
            # Close the browser even if the test fails
            await browser.close()

In [42]:
# test on single track
await get_stream_count_browser(stream_test_df.iloc[1]["spotify_url"])

'2,424,369'

CONCLUSION:

The HTML retrieved using requests did not contain the test track’s play-count element. Playwright successfully retrieved the displayed count as a string, matching the manual observation. The count will need to be converted to an integer for numerical analysis. Further testing is needed to assess reliability across multiple tracks, including tracks with no displayed count, before developing batch collection or an application.

### 10. CREATE A BATCH FUNCTION TO CHECK STREAM COUNTS ON A DATA SAMPLE

Create a batch function that takes a datafame, visits each track URL and returns the results alongside the matching track IDs.

In [47]:
# create a batch function that takes a dataframe, visits each track URL and retrieves the stream count
# for each track, record:
# Stream count as an integer, or a missing value
# Observation date/time
# Status, such as "retrieved" or "timed out"

async def batch_retrieve_stream_counts(df):
    results = []

    for index, row in df.iterrows():
        stream_count = None

        artist_names = ", ".join(
                        artist["name"] for artist in row["artists"]
                    )

        try:
            # Use the Playwright function that worked
            result = await get_stream_count_browser(row["spotify_url"])

            # Remove commas and convert a numeric count to an integer
            cleaned_count = result.strip().replace(",", "")

            if cleaned_count.isdigit():
                stream_count = int(cleaned_count)
                status = "retrieved"
            else:
                # Preserve the message returned by the function
                status = result

        except Exception as e:
            status = f"error: {type(e).__name__}: {e}"

        results.append({
            "track_id": row["id"],
            "track_name": row["name"],
            "artist": artist_names,
            "search_genre": "electronic dance music",
            "stream_count": stream_count,
            "observation_datetime": pd.Timestamp.now(tz="UTC"),
            "status": status
        })

    observations_df = pd.DataFrame(results)

    # Keep whole-number counts while allowing missing values
    observations_df["stream_count"] = observations_df["stream_count"].astype("Int64")

    return observations_df

In [50]:
# test on three tracks

stream_observations_df = await batch_retrieve_stream_counts(
    stream_test_df.head(3)
)

stream_observations_df

,track_id,track_name,artist,stream_count,observation_datetime,status
0,2sxYN584YAgnRwnmIQF0Ye,CRECIENDO DIARIAMENTE - Afro House,Cokito Deep,4639614,2026-11-25 00:51:12.489900+00:00,retrieved
1,65cd1kUvEU9Q0PmYJRc6Uy,Carne y Tambor,"Victor Cardenas, Exotic, Konig Pry, Ricardo RV",2424369,2026-11-25 00:51:15.341401+00:00,retrieved
2,7vY4hbChNHJudQQpPlnL5M,Deeper Love,Double Touch,1711764,2026-11-25 00:51:18.501093+00:00,retrieved


### CONCLUSION:

The batch function successfully retrieved stream counts for all three test tracks, recording track IDs, titles, artists, observation timestamps, and retrieval statuses. Further testing is needed for tracks with no displayed count and larger batches.